# `system_one` example

Asks a trained model named questions about one state through `jev_model.system_one`, in the shape of TypeSafe's
`system_one` API, offline. Two kinds of question:

- `Noul(instructions)`: a yes/no question; the answer is the probability of yes.
- `Choice(instructions, criteria)`: one label out of several. `criteria` maps each label to when it applies, or to
  `None` when the label speaks for itself; the answer is the top label, its confidence and every label's probability.

Every question of one `system_one` call goes to the model in a single batch.

Needs the `model` extra: `uv sync --extra model --extra notebook`.

## Load a model

In [ ]:
from jev_model.constants import RUNS_DIR
from jev_model.system_one import Choice, ChoiceAnswer, JevHolder, Noul, NoulAnswer

# a run directory (its last stage), a stage directory (`RUN/ce`), or a Hugging Face repo id from `jev-model push`
SOURCE = sorted((RUNS_DIR / "pointer").glob("*"))[-1]
# SOURCE = "user/jev-pointer"

holder = JevHolder.from_pretrained(SOURCE)  # device="" picks cuda when there is one; stage="ce" picks a stage
SOURCE

## Yes/no questions

A `Noul` has only instructions. Its options are always `yes` and `no`, whatever the language of the instructions.

In [ ]:
state = (
    "Hi, I ordered a blender two weeks ago (order #4821) and it still hasn't arrived. "
    "The tracking page has said 'in transit' for nine days. I'd like a refund if it can't be delivered by Friday."
)

answers = holder.system_one(
    state,
    {
        "complaint": Noul("Is the customer complaining?"),
        "wants_refund": Noul("Does the customer ask for a refund, at least conditionally?"),
        "gave_order_id": Noul("Does the message include an order number?"),
        "is_spam": Noul("Is this message spam?"),
    },
)
for name, a in answers.items():
    print(f"{name:15} P(yes) = {a.noul:.3f}")

## Choice questions

Each criterion becomes an option `label: description`, or just `label` when the description is `None`.
A choice needs at least two criteria.

In [ ]:
answers = holder.system_one(
    state,
    {
        "topic": Choice(
            "What is the message mainly about?",
            {
                "delivery": "shipping, tracking or a late or lost parcel",
                "billing": "charges, invoices or payment methods",
                "product": "how a product works or a defect in it",
                "account": "logging in, passwords or profile settings",
            },
        ),
        "sentiment": Choice("What is the customer's tone?", {"positive": None, "neutral": None, "negative": None}),
    },
)
for name, a in answers.items():
    probs = ", ".join(f"{label} {p:.2f}" for label, p in a.probabilities.items())
    print(f"{name:10} -> {a.choice} ({a.confidence:.2f})   [{probs}]")

## Mixing both kinds

One call can hold both kinds; answers come back under the same names. Check the type to read them.

In [ ]:
state = (
    "Alice is taller than Bob. Bob is taller than Carol. Dan is shorter than Carol. "
    "Erin is taller than Alice."
)

answers = holder.system_one(
    state,
    {
        "alice_over_dan": Noul("Is Alice taller than Dan?"),
        "carol_over_erin": Noul("Is Carol taller than Erin?"),
        "tallest": Choice("Who is the tallest?", {name: None for name in ["Alice", "Bob", "Carol", "Dan", "Erin"]}),
        "shortest": Choice("Who is the shortest?", {name: None for name in ["Alice", "Bob", "Carol", "Dan", "Erin"]}),
    },
)


def show(answers: dict[str, NoulAnswer | ChoiceAnswer]) -> None:
    for name, a in answers.items():
        if isinstance(a, NoulAnswer):
            print(f"{name:16} yes {a.noul:.3f}")
        else:
            print(f"{name:16} {a.choice} ({a.confidence:.2f})")


show(answers)

## A row from a built dataset

States from `jev prepare` work as they are. Here the dataset's options become `Choice` labels with no
description, so the model reads them exactly as in training and the answer can be checked against the label.

In [ ]:
from datasets import load_from_disk

from jev_model.constants import DATA_DIR

NAME = "qasc"  # any dataset built under DATA_DIR
row = load_from_disk(str(DATA_DIR / NAME))["test"][0]

texts = [option["text"] for option in row["options"]]
# labels are dict keys, so repeated option texts would collapse
assert len(set(texts)) == len(texts)
a = holder.system_one(row["state"], {"q": Choice(row["question"], dict.fromkeys(texts))})["q"]

gold = texts[max(range(len(texts)), key=row["label"].__getitem__)]
print(row["state"], "")
print(row["question"])
print("predicted:", a.choice, f"({a.confidence:.2f})")
print("gold:     ", gold)

## Errors

Questions are checked before anything reaches the model.

In [ ]:
for questions in [{}, {"one": Choice("Pick one", {"only": None})}, {"bad": "Is it raining?"}]:
    try:
        holder.system_one("It is raining.", questions)
    except ValueError as e:
        print("ValueError:", e)